# 1.5 超低比特量化（Sub-2-bit Quantization）

## 什么是超低比特量化？

将权重量化到 **2-bit 及以下**（含 1.58-bit 三值 BitNet），目标是把 7B 级模型压到约 1–2GB，让中低端手机也能流畅运行。

## 为什么要学这一节？

1. INT4 已接近成熟，下一阶段压缩红利来自 2-bit / 1.58-bit。
2. BitNet 把 MatMul 退化成加减法，对 NPU/DSP 友好。
3. 产业级 2-bit（如 HY-1.8B-2Bit）已落地，QAT 成为标配。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

## 1.5.1 BitNet 1.58-bit（三值量化）

权重仅取 $\{-1, 0, +1\}$，信息量约 $\log_2 3 \approx 1.585$ bit。前向可用 `sign` + 稀疏掩码近似：

$$W_q = \mathrm{RoundClip}\left(\frac{W}{\gamma + \epsilon}, -1, 1\right),\quad \gamma=\frac{1}{nm}\|W\|_1$$

In [ ]:
def absmean_scale(w: torch.Tensor) -> torch.Tensor:
    """BitNet AbsMean 缩放因子 γ = mean(|W|)"""
    return w.abs().mean().clamp(min=1e-8)


def ternary_quantize(w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """将权重量化到 {-1, 0, +1}，并返回缩放因子。"""
    gamma = absmean_scale(w)
    w_scaled = (w / gamma).clamp(-1, 1)
    # 简单阈值：|x|<0.5 → 0，否则 sign
    w_q = torch.where(w_scaled.abs() < 0.5, torch.zeros_like(w_scaled), w_scaled.sign())
    return w_q, gamma


def ternary_matmul(x: torch.Tensor, w_q: torch.Tensor, gamma: torch.Tensor) -> torch.Tensor:
    """三值权重矩阵乘：x @ (gamma * W_q)^T ≈ 加减法 + 一次缩放。"""
    # 拆成正负掩码，避免真正乘法（教学演示）
    pos = (w_q > 0).to(x.dtype)
    neg = (w_q < 0).to(x.dtype)
    y = x @ pos.T - x @ neg.T
    return y * gamma


# 演示：与 FP 基线对比
d_in, d_out, bsz = 256, 128, 8
W = torch.randn(d_out, d_in) * 0.05
X = torch.randn(bsz, d_in)
W_q, gamma = ternary_quantize(W)
y_fp = X @ W.T
y_t = ternary_matmul(X, W_q, gamma)
rel_err = (y_fp - y_t).norm() / y_fp.norm()
sparsity = (W_q == 0).float().mean().item()
bits_per_weight = np.log2(3)
print("=== BitNet-style 1.58-bit ===")
print(f"相对误差: {rel_err.item():.4f}")
print(f"零值稀疏度: {sparsity*100:.1f}%")
print(f"等效比特: {bits_per_weight:.3f}")
print(f"相对 FP16 体积: {bits_per_weight/16*100:.1f}%")

## 1.5.2 2-bit 分组量化 + 残差补偿

产业级 2-bit 通常采用：**分组量化（group size=64）+ 非均匀码本 + 残差补偿**。下面用均匀 2-bit + 简单残差演示核心思想。

In [ ]:
def group_quantize_2bit(w: torch.Tensor, group_size: int = 64):
    """逐组均匀 2-bit 量化，返回量化索引与每组 scale。"""
    out, inn = w.shape
    assert inn % group_size == 0
    w_g = w.view(out, -1, group_size)
    # 对称量化到 {-1.5,-0.5,0.5,1.5} 四个档（映射到 0..3）
    levels = torch.tensor([-1.5, -0.5, 0.5, 1.5])
    scales = w_g.abs().amax(dim=-1, keepdim=True).clamp(min=1e-8) / 1.5
    w_n = w_g / scales
    # 最近邻到 4 个 level
    dist = (w_n.unsqueeze(-1) - levels).abs()
    idx = dist.argmin(dim=-1)  # [out, n_groups, group]
    w_hat = levels[idx] * scales
    return idx, scales.squeeze(-1), w_hat.view_as(w)


def residual_2bit(w: torch.Tensor, group_size: int = 64):
    """一阶残差补偿：量化 W，再量化残差 R=W-Ŵ，推理时相加。"""
    idx1, s1, w1 = group_quantize_2bit(w, group_size)
    resid = w - w1
    idx2, s2, w2 = group_quantize_2bit(resid, group_size)
    return w1 + w2, (idx1, s1, idx2, s2)


W = torch.randn(64, 256) * 0.08
_, _, W2 = group_quantize_2bit(W)
W2r, _meta = residual_2bit(W)
err2 = (W - W2).norm() / W.norm()
err2r = (W - W2r).norm() / W.norm()
print("=== 2-bit group quant ===")
print(f"均匀 2-bit 相对误差: {err2.item():.4f}")
print(f"+残差补偿相对误差: {err2r.item():.4f}")
print(f"体积(相对 FP16): 2/16 = 12.5%（残差版约 25%，精度明显更好）")

## 1.5.3 QAT 伪量化节点（端侧训练标准实践）

2026 年起，Gemma 等模型原生 QAT：训练时插入 fake-quant，推理时换成真量化。

In [ ]:
class FakeQuant2Bit(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, group_size=64):
        _, _, w_hat = group_quantize_2bit(w, group_size)
        return w_hat

    @staticmethod
    def backward(ctx, grad_output):
        # STE：梯度直通
        return grad_output, None


class TinyLinearQAT(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)

    def forward(self, x):
        w_q = FakeQuant2Bit.apply(self.weight, 64)
        return F.linear(x, w_q)


model = TinyLinearQAT(256, 128)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
x = torch.randn(32, 256)
target = torch.randn(32, 128)
losses = []
for step in range(30):
    opt.zero_grad()
    loss = F.mse_loss(model(x), target)
    loss.backward()
    opt.step()
    losses.append(loss.item())
print("=== 2-bit QAT STE 演示 ===")
print(f"loss: {losses[0]:.4f} → {losses[-1]:.4f}")
print("要点: 前向量化、反向 STE；产业实现还需校准集与敏感层保护。")

## 小结与部署建议

| 方案 | 比特 | 典型体积(7B) | 硬件需求 | 推荐场景 |
|------|------|-------------|---------|---------|
| BitNet 1.58-bit | ~1.58 | ~1.4GB | 原生三值加速最佳 | 新芯片 / 极致压缩 |
| 2-bit + 残差 | 2(+2) | ~1.8–3.5GB | 通用 INT 核 | 手机端产业落地 |
| INT4 QAT | 4 | ~3.5GB | 广泛支持 | 当前主力 |

**端侧启示**：优先选原生 QAT 发布的小模型；对 FP16 做极低比特 PTQ 精度往往不够。